# Desproporcionalidade

In [1]:
import sys
sys.path.append('../../src/')


%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 100
%config SqlMagic.named_parameters = "enabled"
%config SqlMagic.autocommit = True

# Banco em arquivo (persiste durante a sessão).
#%sql duckdb:///content/vigimed.duckdb
# Se preferir memória: 
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [2]:
%sql ROLLBACK;

Running query in 'duckdb:///:memory:'

,Success


In [3]:
%%sql
DROP TABLE IF EXISTS dim_atc;
CREATE TABLE dim_atc AS
SELECT * FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\dim_atc\dim_atc.parquet');


DROP TABLE IF EXISTS fat_medicamentos;
CREATE TABLE fat_medicamentos AS
SELECT * FROM read_parquet('C:\Users\silma\Projetos\vigimed\data\03_gold\fat_medicamentos\fat_medicamentos.parquet');

Running query in 'duckdb:///:memory:'

,Success


In [4]:
%%sql 
select * from fat_medicamentos
LIMIT 5

Running query in 'duckdb:///:memory:'

,IDENTIFICACAO_NOTIFICACAO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,ATC_CODE_LEVEL_4,DETENTOR_REGISTRO,CONCENTRACAO,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,...,VIA_ADMINISTRACAO_CHAVE_VALOR,VIA_ADMINISTRACAO_MAE_PAI_VALOR,CONCENTRACAO_TIPO_CHAVE,CONCENTRACAO_TIPO_VALOR,CONCENTRACAO_VALOR,CONCENTRACAO_VALOR_NUMERADOR,CONCENTRACAO_VALOR_DENOMINADOR,DOSE_TIPO_CHAVE,DOSE_TIPO_VALOR,DOSE_VALOR
0,BR-ANVISA-300000004,OXACILINA,OXACILLIN SODIUM,J01CF,TEUTO,None,NaN,None,None,250 MILLIGRAM (MG),...,6,5,desconhecido,0,NaN,NaN,NaN,milligram (mg),1,250.0
1,BR-ANVISA-300000005,PARACEMOL,PARACETAMOL,N02BE,None,None,NaN,None,None,None,...,5,5,desconhecido,0,NaN,NaN,NaN,desconhecido,0,NaN
2,BR-ANVISA-300000007,DIAMOX,ACETAZOLAMIDE SODIUM,C03,None,None,NaN,None,None,250 MILLIGRAM (MG),...,5,5,desconhecido,0,NaN,NaN,NaN,milligram (mg),1,250.0
3,BR-ANVISA-300000007,DIAMOX,ACETAZOLAMIDE SODIUM,N03AX,None,None,NaN,None,None,250 MILLIGRAM (MG),...,5,5,desconhecido,0,NaN,NaN,NaN,milligram (mg),1,250.0
4,BR-ANVISA-300000007,DIAMOX,ACETAZOLAMIDE SODIUM,S01EC,None,None,NaN,None,None,250 MILLIGRAM (MG),...,5,5,desconhecido,0,NaN,NaN,NaN,milligram (mg),1,250.0


In [5]:
%%sql 
select * from dim_atc
where ATC_CODE_LEVEL_4 ='L04AB'

Running query in 'duckdb:///:memory:'

,ATC_CODE_LEVEL_1,ATC_CODE_LEVEL_2,ATC_CODE_LEVEL_3,ATC_CODE_LEVEL_4,ATC_CODE_LEVEL_5,ATC_LEVEL,ATC_CODE_LEVEL_1_LEVEL_NAME,ATC_CODE_LEVEL_2_LEVEL_NAME,ATC_CODE_LEVEL_3_LEVEL_NAME,ATC_CODE_LEVEL_4_LEVEL_NAME,ATC_CODE_LEVEL_5_LEVEL_NAME,ATC_CODE_LEVEL,ATC_NAME_LEVEL,DDD_VALUE,UNIT,ADM_R,COMMENT
0,L,L04,L04A,L04AB,L04AB01,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,ETANERCEPT,L04AB01,etanercept,7.00,mg,P,None
1,L,L04,L04A,L04AB,L04AB02,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,INFLIXIMAB,L04AB02,infliximab,3.75,mg,P,None
2,L,L04,L04A,L04AB,L04AB03,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,AFELIMOMAB,L04AB03,afelimomab,NaN,None,None,None
3,L,L04,L04A,L04AB,L04AB04,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,ADALIMUMAB,L04AB04,adalimumab,2.90,mg,P,None
4,L,L04,L04A,L04AB,L04AB05,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,CERTOLIZUMAB PEGOL,L04AB05,certolizumab pegol,14.00,mg,P,None
5,L,L04,L04A,L04AB,L04AB06,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,GOLIMUMAB,L04AB06,golimumab,1.66,mg,P,None
6,L,L04,L04A,L04AB,L04AB07,5,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,OPINERCEPT,L04AB07,opinercept,7.00,mg,P,None
7,L,L04,L04A,L04AB,None,4,ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS,IMMUNOSUPPRESSANTS,IMMUNOSUPPRESSANTS,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,None,L04AB,Tumor necrosis factor alpha (TNF-alpha) inhibi...,NaN,None,None,None


In [7]:
%%sql 

with fat_med as (
    select fat_medicamentos.*,
    (case 
    when atc.ATC_CODE_LEVEL_4 in ('L04AB', 'L01FA', 'L04AC','L04AF') 
        then concat(atc.ATC_CODE_LEVEL_4,'_',atc.ATC_CODE_LEVEL_4_LEVEL_NAME) 
        else 'Outros' 
        end) as ATC_CODE_LEVEL_4_LEVEL_NAME
    from fat_medicamentos
    inner join dim_atc as atc
    on fat_medicamentos.ATC_CODE_LEVEL_4 = atc.ATC_CODE_LEVEL_4 and 
    atc.ATC_CODE_LEVEL_5 is null
)

select
ATC_CODE_LEVEL_4_LEVEL_NAME
,count(distinct(IDENTIFICACAO_NOTIFICACAO)) AS qt_notificacoes
from fat_med
group by 1
ORDER BY 1

Running query in 'duckdb:///:memory:'

,ATC_CODE_LEVEL_4_LEVEL_NAME,qt_notificacoes
0,L01FA_CD20 (CLUSTERS OF DIFFERENTIATION 20) IN...,3114
1,L04AB_TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) ...,17261
2,L04AC_INTERLEUKIN INHIBITORS,5672
3,L04AF_JANUS-ASSOCIATED KINASE (JAK) INHIBITORS,416
4,Outros,271583


In [16]:
%%sql
DROP TABLE IF EXISTS fat_med_w_atc_level_5;

CREATE TABLE fat_med_w_atc_level_5 AS
with fat_med as (
SELECT 
    fat_medicamentos.*, atc.ATC_CODE_LEVEL_4_LEVEL_NAME,
    (
        CASE 
            WHEN atc.ATC_CODE_LEVEL_4 IN ('L04AB', 'L01FA', 'L04AC','L04AF') 
                THEN (
                    CASE 
                        /*Rituximab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%rituximab%' THEN 'L01FA01_Rituximab'
                        /*Abatacept*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%abatacept%' THEN 'L04AA24_Abatacept'
                        /*Etanercept*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%etanercept%' THEN 'L04AB01_Etanercept'
                        /*Infliximab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%infliximab%' THEN 'L04AB02_Infliximab'
                        /*Adalimumab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%adalimumab%' THEN 'L04AB04_Adalimumab'
                        /*Certolizumab Pegol*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumab%' THEN 'L04AB05_Certolizumab Pegol'   
                        -- WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumab pegol%' THEN 'L04AB05_Certolizumab Pegol'
                        -- WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumabe pegol%' THEN 'L04AB05_Certolizumab Pegol'
                        
                        /*Golimumab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%golimumab%' THEN 'L04AB06_Golimumab'
                        /*Tocilizumab*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%tocilizumab%' THEN 'L04AC07_Tocilizumab'
                        /*Baricitinib */
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%baricitinib%' THEN 'L04AF02_Baricitinib'
                        /*Upadacitinib*/
                        WHEN LOWER(PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%upadacitinib%' THEN 'L04AF03_Upadacitinib' 
                        ELSE 'missing'
                    END
                )
            ELSE 'Outros'
        END
    ) AS PRINCIPIOS_ATIVOS_CLEAN
FROM fat_medicamentos
INNER JOIN dim_atc AS atc
    ON fat_medicamentos.ATC_CODE_LEVEL_4 = atc.ATC_CODE_LEVEL_4 
    AND atc.ATC_LEVEL = 4
)
select 
    (case when PRINCIPIOS_ATIVOS_CLEAN in ('missing','Outros') then 'Outros' 
        else concat(ATC_CODE_LEVEL_4,'_',ATC_CODE_LEVEL_4_LEVEL_NAME) end) as ATC_LEVEL_4_NAME
,*
from fat_med

Running query in 'duckdb:///:memory:'

,Success


In [17]:
%%sql
select
ATC_LEVEL_4_NAME
,PRINCIPIOS_ATIVOS_CLEAN
,count(distinct(IDENTIFICACAO_NOTIFICACAO)) AS qt_notificacoes
from fat_med_w_atc_level_5
group by 1,2
ORDER BY 1,2

Running query in 'duckdb:///:memory:'

,ATC_LEVEL_4_NAME,PRINCIPIOS_ATIVOS_CLEAN,qt_notificacoes
0,L01FA_CD20 (CLUSTERS OF DIFFERENTIATION 20) IN...,L01FA01_Rituximab,2993
1,L04AB_TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) ...,L04AB01_Etanercept,372
2,L04AB_TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) ...,L04AB02_Infliximab,10529
3,L04AB_TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) ...,L04AB04_Adalimumab,1616
4,L04AB_TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) ...,L04AB05_Certolizumab Pegol,1221
5,L04AB_TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) ...,L04AB06_Golimumab,3743
6,L04AC_INTERLEUKIN INHIBITORS,L04AC07_Tocilizumab,854
7,L04AF_JANUS-ASSOCIATED KINASE (JAK) INHIBITORS,L04AF02_Baricitinib,94
8,L04AF_JANUS-ASSOCIATED KINASE (JAK) INHIBITORS,L04AF03_Upadacitinib,267
9,Outros,Outros,271583


In [18]:
%%sql
select PRINCIPIOS_ATIVOS_WHODRUG
,count(distinct(IDENTIFICACAO_NOTIFICACAO)) AS qt_notificacoes
from fat_med_w_atc_level_5 
where ATC_CODE_LEVEL_4='L04AC' and PRINCIPIOS_ATIVOS_CLEAN='missing'
group by 1
order by 2 desc
limit 10

Running query in 'duckdb:///:memory:'

,PRINCIPIOS_ATIVOS_WHODRUG,qt_notificacoes
0,SECUKINUMAB,1487
1,USTEKINUMAB,982
2,USTEQUINUMABE,817
3,SECUQUINUMABE,552
4,RISANKIZUMAB,193
5,RISANQUIZUMABE,159
6,IXEQUIZUMABE,150
7,GUSELCUMABE,131
8,RISANKIZUMAB RZAA,115
9,GUSELKUMAB,107


In [19]:
%%sql
select PRINCIPIOS_ATIVOS_WHODRUG
,count(distinct(IDENTIFICACAO_NOTIFICACAO)) AS qt_notificacoes
from fat_med_w_atc_level_5 
where ATC_CODE_LEVEL_4='L01FA' and PRINCIPIOS_ATIVOS_CLEAN='missing'
group by 1
order by 2 desc 

Running query in 'duckdb:///:memory:'

,PRINCIPIOS_ATIVOS_WHODRUG,qt_notificacoes
0,OBINUTUZUMABE,75
1,OBINUTUZUMAB,46
2,OFATUMUMAB,5


In [20]:
%%sql
select PRINCIPIOS_ATIVOS_WHODRUG
,count(distinct(IDENTIFICACAO_NOTIFICACAO)) AS qt_notificacoes
from fat_med_w_atc_level_5 
where ATC_CODE_LEVEL_4='L04AB' and PRINCIPIOS_ATIVOS_CLEAN='missing'
group by 1
order by 2 desc 

Running query in 'duckdb:///:memory:'

,PRINCIPIOS_ATIVOS_WHODRUG,qt_notificacoes


In [21]:
%%sql
select PRINCIPIOS_ATIVOS_WHODRUG
,count(distinct(IDENTIFICACAO_NOTIFICACAO)) AS qt_notificacoes
from fat_med_w_atc_level_5 
where ATC_CODE_LEVEL_4='L04AF' and PRINCIPIOS_ATIVOS_CLEAN='missing'
group by 1
order by 2 desc 

Running query in 'duckdb:///:memory:'

,PRINCIPIOS_ATIVOS_WHODRUG,qt_notificacoes
0,CITRATO DE TOFACITINIBE,37
1,TOFACITINIBE,18
2,RITLECITINIB TOSILATE,2
3,RUXOLITINIBE,1
4,DEUCRAVACITINIBE,1


## Analytic Query